# Title + Objective (Markdown)

Hyperparameter tuning with correct CV + no leakage

# Hyperparameter Tuning — EcoType Forest Cover Classification

## Objective
- tune the 1–2 best-performing models
- compare tuned performance vs baseline
- use leakage-safe pipelines and imbalance handling helpers
- save tuning outputs for later evaluation and deployment

# Import + Config

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml
import joblib

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold

project_root = Path.cwd().resolve().parents[0]
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.modeling.imbalance import build_imbalanced_pipeline, get_imbalance_config
from src.modeling.tune import run_tuning, load_model_param_grid

seed = 42
primary_metric = "f1_macro"

config_dir = project_root / "config"
paths_file = config_dir / "paths.yaml"
train_file = config_dir / "train.yaml"
model_grid_file= config_dir / "model_grid.yaml"

with open(paths_file, "r", encoding="utf-8") as f:
    paths_cfg = yaml.safe_load(f)

with open(train_file, "r", encoding="utf-8") as f:
    train_cfg = yaml.safe_load(f)

data_dir = project_root / paths_cfg["data"]["processed_dir"]
report_dir = project_root / paths_cfg["artifacts"]["reports_dir"] / "model_results"
fig_dir = project_root / paths_cfg["artifacts"]["figures_dir"]
tuning_dir = project_root / "artifacts" / "tuning"

report_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)
tuning_dir.mkdir(parents=True, exist_ok=True)

# Load Data (Code)

Load X_train, y_train

In [2]:
train_features_file = project_root / paths_cfg["files"]["train_features"]
test_features_file = project_root / paths_cfg["files"]["test_features"]
train_labels_file = project_root / paths_cfg["files"]["train_labels"]
test_labels_file = project_root / paths_cfg["files"]["test_labels"]
preprocessor_file = project_root / "models" / "preprocessing_pipeline.joblib"

X_train = pd.read_csv(train_features_file)
X_test = pd.read_csv(test_features_file)
y_train = pd.read_csv(train_labels_file).squeeze("columns")
y_test = pd.read_csv(test_labels_file).squeeze("columns")
preprocessor = joblib.load(preprocessor_file)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)
print("Preprocessor loaded:", type(preprocessor).__name__)

X_train: (116712, 20)
X_test : (29178, 20)
y_train: (116712,)
y_test : (29178,)
Preprocessor loaded: ColumnTransformer


## Set CV + Imbalance Strategy

In [3]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=seed,
)

imbalance_strategy = "class_weight"   # change to "smote" if that won in notebook 06
imbalance_notes = get_imbalance_config(imbalance_strategy)
imbalance_notes

{'strategy': 'class_weight', 'class_weight': 'balanced'}

# Define Pipeline (Code)

preprocessor + model

optional imbalance step (SMOTE inside pipeline)

In [4]:
rf = RandomForestClassifier(
    random_state=seed,
    n_jobs=1,
)

et = ExtraTreesClassifier(
    random_state=seed,
    n_jobs=1,
)

rf_pipeline = build_imbalanced_pipeline(
    preprocessor=preprocessor,
    estimator=rf,
    strategy = imbalance_strategy,
    class_weight="balanced",
    random_state=seed,
)

et_pipeline = build_imbalanced_pipeline(
    preprocessor=preprocessor,
    estimator=et,
    strategy=imbalance_strategy,
    class_weight="balanced",
    random_state=seed,
)

# Run Tuning (Code)

In [5]:
import importlib
import src.modeling.tune as tune

importlib.reload(tune)
print(tune.__file__)
print(tune.load_model_param_grid("random_forest", model_grid_file))

F:\DATA SCIENCE\Projects\Forest Cover Type Prediction\src\modeling\tune.py
{'model__n_estimators': [100, 150, 200], 'model__max_depth': [15, 20, None], 'model__min_samples_split': [2, 5], 'model__min_samples_leaf': [1, 2], 'model__max_features': ['sqrt']}


In [6]:
rf_tuning = run_tuning(
    model_name="random_forest",
    pipeline=rf_pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    config_path=model_grid_file,
    artifacts_dir=tuning_dir,
    scoring=primary_metric,
    search_type="random",
    n_jobs=1,
    n_iter=8,
    verbose=2,
    random_state=seed,
)

rf_tuning["best_score"], rf_tuning["best_params"]

Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=2, model__min_samples_split=5, model__n_estimators=150; total time=  39.9s
[CV] END model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=2, model__min_samples_split=5, model__n_estimators=150; total time=  39.7s
[CV] END model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=2, model__min_samples_split=5, model__n_estimators=150; total time=  41.4s
[CV] END model__max_depth=20, model__max_features=sqrt, model__min_samples_leaf=2, model__min_samples_split=5, model__n_estimators=100; total time=  26.3s
[CV] END model__max_depth=20, model__max_features=sqrt, model__min_samples_leaf=2, model__min_samples_split=5, model__n_estimators=100; total time=  26.6s
[CV] END model__max_depth=20, model__max_features=sqrt, model__min_samples_leaf=2, model__min_samples_split=5, model__n_estimators=100; total time=  26.8s
[CV

(0.8949366792687229,
 {'model__n_estimators': 100,
  'model__min_samples_split': 2,
  'model__min_samples_leaf': 1,
  'model__max_features': 'sqrt',
  'model__max_depth': None})

In [7]:
et_tuning = run_tuning(
    model_name="extra_trees",
    pipeline=et_pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    config_path=model_grid_file,
    artifacts_dir=tuning_dir,
    scoring=primary_metric,
    search_type="random",
    n_jobs=1,
    n_iter=8,
    verbose=2,
    random_state=seed,
)

et_tuning["best_score"], et_tuning["best_params"]

Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END model__max_depth=20, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=5, model__n_estimators=150; total time=  15.6s
[CV] END model__max_depth=20, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=5, model__n_estimators=150; total time=  15.2s
[CV] END model__max_depth=20, model__max_features=sqrt, model__min_samples_leaf=1, model__min_samples_split=5, model__n_estimators=150; total time=  15.9s
[CV] END model__max_depth=15, model__max_features=sqrt, model__min_samples_leaf=2, model__min_samples_split=2, model__n_estimators=100; total time=   8.0s
[CV] END model__max_depth=15, model__max_features=sqrt, model__min_samples_leaf=2, model__min_samples_split=2, model__n_estimators=100; total time=   8.3s
[CV] END model__max_depth=15, model__max_features=sqrt, model__min_samples_leaf=2, model__min_samples_split=2, model__n_estimators=100; total time=   8.4s
[CV] END 

(0.8917623153415942,
 {'model__n_estimators': 150,
  'model__min_samples_split': 2,
  'model__min_samples_leaf': 1,
  'model__max_features': 'sqrt',
  'model__max_depth': None})

# Extract Results (Code)

best_params_

best_score_

cv_results_ dataframe

In [8]:
def evaluate_model(name, estimator, X_test, y_test):
    y_pred = estimator.predict(X_test)
    return {
        "model": name,
        "accuracy_test": accuracy_score(y_test, y_pred),
        "f1_macro_test": f1_score(y_test, y_pred, average="macro"),
    }

results_tuned = pd.DataFrame([
    evaluate_model("random_forest_tuned", rf_tuning["best_estimator"], X_test, y_test),
    evaluate_model("extra_trees_tuned", et_tuning["best_estimator"], X_test, y_test),
])

results_tuned

,model,accuracy_test,f1_macro_test
0,random_forest_tuned,0.959387,0.915157
1,extra_trees_tuned,0.953629,0.908029


In [9]:
baseline_results = pd.DataFrame([
    {"model": "random_forest_baseline", "accuracy_test": np.nan, "f1_macro_test": 0.91145},
    {"model": "extra_trees_baseline", "accuracy_test": np.nan, "f1_macro_test": 0.91134},
])

comparison_df = pd.concat([baseline_results, results_tuned], ignore_index=True)
comparison_df

,model,accuracy_test,f1_macro_test
0,random_forest_baseline,NaN,0.911450
1,extra_trees_baseline,NaN,0.911340
2,random_forest_tuned,0.959387,0.915157
3,extra_trees_tuned,0.953629,0.908029


# Save Tuning Results (Code)

Save tuning_results.csv

Save best estimator joblib

In [12]:
tuning_results = pd.DataFrame([
    {
        "model": "random_forest",
        "search_type": "random",
        "cv_best_score": rf_tuning["best_score"],
        "test_f1_macro": results_tuned.loc[results_tuned["model"] == "random_forest_tuned", "f1_macro_test"].iloc[0],
        "best_params": str(rf_tuning["best_params"]),
        "imbalance_strategy": imbalance_strategy,
    },
    {
        "model": "extra_trees",
        "search_type": "random",
        "cv_best_score": et_tuning["best_score"],
        "test_f1_macro": results_tuned.loc[results_tuned["model"] == "extra_trees_tuned", "f1_macro_test"].iloc[0],
        "best_params": str(et_tuning["best_params"]),
        "imbalance_strategy": imbalance_strategy,
    },
])

tuning_results_file = report_dir / "tuning_results.csv"
tuning_results.to_csv(tuning_results_file, index=False)

print("Saved:", tuning_results_file)
tuning_results

Saved: F:\DATA SCIENCE\Projects\Forest Cover Type Prediction\reports\model_results\tuning_results.csv


,model,search_type,cv_best_score,test_f1_macro,best_params,imbalance_strategy
0,random_forest,random,0.894937,0.915157,"{'model__n_estimators': 100, 'model__min_sampl...",class_weight
1,extra_trees,random,0.891762,0.908029,"{'model__n_estimators': 150, 'model__min_sampl...",class_weight


In [13]:
rf_top = rf_tuning["cv_results_df"].sort_values("rank_test_score").head(10)
et_top = et_tuning["cv_results_df"].sort_values("rank_test_score").head(10)

rf_top[["rank_test_score", "mean_test_score", "std_test_score", "params"]].head()

,rank_test_score,mean_test_score,std_test_score,params
6,1,0.894937,0.003756,"{'model__n_estimators': 100, 'model__min_sampl..."
3,2,0.894921,0.003534,"{'model__n_estimators': 200, 'model__min_sampl..."
0,3,0.890687,0.003618,"{'model__n_estimators': 150, 'model__min_sampl..."
9,4,0.889125,0.003166,"{'model__n_estimators': 200, 'model__min_sampl..."
5,5,0.888645,0.002181,"{'model__n_estimators': 200, 'model__min_sampl..."


# Notes (Markdown)

tuned vs baseline improvement

decision: final model chosen